In [ ]:
!pip install pandas numpy matplotlib seaborn scipy nbformat plotly ipywidgets

In [ ]:
# ==================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from collections import Counter
from itertools import combinations
import scipy
import nbformat
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, HTML
import pandas as pd
import re

# Set plotting style
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 100

# Nootropic & Supplement Sentiment Analysis
This notebook analyzes the output of the `main.py` script. The input file (`ref_res.jsonl`) contains one JSON object per *sentence* that mentions at least one supplement.
I will explore:
1.  **High-Level Statistics & Normalization:** Overall data shape, classification, normalization.
2.  **Supplement Frequency:** Which supplements are mentioned most often?
3.  **Supplement Sentiment:** Which supplements are discussed most positively, negatively, or as "ineffective"?
4.  **Stack Analysis:** What are the most common supplement co-mentions (stacks)?
5.  **Aspect-Based Analysis:** What aspects (e.g., "cognition", "sleep", "side_effects") are discussed, and for which supplements?
6.  **Qualitative Review:** A look at the raw sentences for specific topics.

In [ ]:
final_data_path = 'data_for_analysis_final.jsonl'

df = pd.read_json(final_data_path, lines=True, encoding='utf-8')

In [ ]:
print("\n--- Data Body ---")
display(df.sample(20))
print("\n--- Data Info ---")
df.info()

In [ ]:
supplement_mapping = {
    # --- Generic Terms ---
    'supplements': 'Supplement',
    'supplement': 'Supplement',

    # --- Vitamins ---
    'multivitamin': 'Multivitamin',
    'vitamin d': 'Vitamin D',
    'vitamin d3': 'Vitamin D',
    'vitamin c': 'Vitamin C',
    'vitamin b12': 'Vitamin B12',
    'b12': 'Vitamin B12',
    'b-complex': 'B-Complex',
    'b - complex': 'B-Complex',
    'vitamin k2': 'Vitamin K2',
    'vitamin b3': 'Vitamin B3',
    'vitamin b9': 'Vitamin B9',
    'vitamin b6': 'Vitamin B6',
    'vitamin a': 'Vitamin A',
    'vitamin e': 'Vitamin E',
    'vitamin b1': 'Vitamin B1',
    'vitamin b5': 'Vitamin B5',
    'vitamin b2': 'Vitamin B2',
    'vitamin b7': 'Vitamin B7',

    # --- Minerals ---
    'magnesium': 'Magnesium',
    'magnesium glycinate': 'Magnesium Glycinate',
    'magnesium citrate': 'Magnesium Citrate',
    'magnesium threonate': 'Magnesium Threonate',
    'zinc': 'Zinc',
    'iron': 'Iron',
    'copper': 'Copper',
    'iodine': 'Iodine',
    'selenium': 'Selenium',
    'manganese': 'Manganese',

    # --- Amino Acids & Derivatives ---
    'creatine': 'Creatine',
    'creatine monohydrate': 'Creatine',
    'l-theanine': 'L-Theanine',
    'theanine': 'L-Theanine',
    'l - theanine': 'L-Theanine',
    'taurine': 'Taurine',
    'n-acetyl cysteine': 'NAC',
    'nac': 'NAC',
    'n - acetyl cysteine': 'NAC',
    'gaba': 'GABA',
    'tyrosine': 'L-Tyrosine',
    'l-tyrosine': 'L-Tyrosine',
    'l - tyrosine': 'L-Tyrosine',
    'nalt': 'NALT (N-Acetyl L-Tyrosine)',
    'beta-alanine': 'Beta-Alanine',
    'beta - alanine': 'Beta-Alanine',
    'glycine': 'Glycine',
    'acetyl-l-carnitine': 'Acetyl-L-Carnitine',
    'alcar': 'Acetyl-L-Carnitine',
    'acetyl - l - carnitine': 'Acetyl-L-Carnitine',
    'l-carnitine': 'L-Carnitine',
    'l - carnitine': 'L-Carnitine',
    'glutamine': 'Glutamine',
    '5-htp': '5-HTP',
    '5 - htp': '5-HTP',
    'l-tryptophan': 'L-Tryptophan',
    'l - tryptophan': 'L-Tryptophan',
    'citrulline malate': 'Citrulline Malate',
    'l-citrulline': 'L-Citrulline',
    'l - citrulline': 'L-Citrulline',
    'l-arginine': 'L-Arginine',
    'l - arginine': 'L-Arginine',
    'phenylalanine': 'Phenylalanine',
    'dlpa': 'Phenylalanine',

    # --- Fungi (Medicinal Mushrooms) ---
    "lion's mane": "Lion's Mane",
    'lions mane': "Lion's Mane",
    "lion 's mane": "Lion's Mane",
    'hericium erinaceus': "Lion's Mane",
    'cordyceps': 'Cordyceps',
    'reishi': 'Reishi',
    'chaga': 'Chaga',

    # --- Herbs & Adaptogens ---
    'ashwagandha': 'Ashwagandha',
    'ksm-66': 'Ashwagandha',
    'sensoril': 'Ashwagandha',
    'curcumin': 'Turmeric/Curcumin',
    'turmeric': 'Turmeric/Curcumin',
    'ginseng': 'Ginseng',
    'panax ginseng': 'Ginseng',
    'siberian ginseng': 'Ginseng',
    'american ginseng': 'Ginseng',
    'rhodiola rosea': 'Rhodiola Rosea',
    'rhodiola': 'Rhodiola Rosea',
    'bacopa monnieri': 'Bacopa Monnieri',
    'bacopa': 'Bacopa Monnieri',
    'holy basil': 'Holy Basil (Tulsi)',
    'tulsi': 'Holy Basil (Tulsi)',
    'tongkat ali': 'Tongkat Ali',
    'fadogia agrestis': 'Fadogia Agrestis',
    'shilajit': 'Shilajit',
    'lemon balm': 'Lemon Balm',
    'ginkgo biloba': 'Ginkgo Biloba',
    'gotu kola': 'Gotu Kola',
    'green tea extract': 'Green Tea Extract',
    'mucuna pruriens': 'Mucuna Pruriens',
    'chamomile': 'Chamomile',
    "st. john's wort": "St. John's Wort",
    "st.john's wort": "St. John's Wort",
    'valerian root': 'Valerian Root',
    'kava': 'Kava',
    'lavender': 'Lavender',
    'sage': 'Sage',
    'yerba mate': 'Yerba Mate',
    'passionflower': 'Passionflower',
    'guarana': 'Guarana',
    'polygala tenuifolia': 'Polygala Tenuifolia',
    'polygala': 'Polygala Tenuifolia',

    # --- Choline Sources ---
    'alpha-gpc': 'Alpha-GPC',
    'alpha gpc': 'Alpha-GPC',
    'alpha - gpc': 'Alpha-GPC',
    'citicoline': 'Citicoline (CDP-Choline)',
    'cdp-choline': 'Citicoline (CDP-Choline)',
    'phosphatidylcholine': 'Phosphatidylcholine',
    'choline bitartrate': 'Choline Bitartrate',

    # --- Racetams & Nootropic Compounds ---
    'nootropics': 'Nootropic (General)',
    'nootropic': 'Nootropic (General)',
    'phenibut': 'Phenibut',
    'noopept': 'Noopept',
    'phenylpiracetam': 'Phenylpiracetam',
    'piracetam': 'Piracetam',
    'aniracetam': 'Aniracetam',
    'fasoracetam': 'Fasoracetam',
    'oxiracetam': 'Oxiracetam',
    'pramiracetam': 'Pramiracetam',
    'coluracetam': 'Coluracetam',
    'nefiracetam': 'Nefiracetam',
    'semax': 'Semax',
    'selank': 'Selank',
    'dihexa': 'Dihexa',
    'cerebrolysin': 'Cerebrolysin',

    # --- Other Compounds ---
    'caffeine': 'Caffeine',
    'fish oil': 'Omega-3',
    'omega-3': 'Omega-3',
    'krill oil': 'Omega-3',
    'melatonin': 'Melatonin',
    'coq10': 'CoQ10',
    'nicotine': 'Nicotine',
    'pregnenolone': 'Pregnenolone',
    'dhea': 'DHEA',
    'alpha lipoic acid': 'Alpha Lipoic Acid',
    'resveratrol': 'Resveratrol',
    'betaine': 'Betaine',
    'modafinil': 'Modafinil',
    'armodafinil': 'Modafinil',
    'adrafinil': 'Adrafinil',
    'inositol': 'Inositol',
    'uridine': 'Uridine',
    'uridine monophosphate': 'Uridine',
    'phosphatidylserine': 'Phosphatidylserine',
    'mct oil': 'MCT Oil',
    'theobromine': 'Theobromine',
    'pqq': 'PQQ',

    # --- Branded Stacks ---
    'alpha brain': 'Alpha Brain',
    'ciltep': 'Ciltep',
}

In [ ]:
df['supplements'] = df['supplements'].apply(
    lambda sup_list: list(set(
        supplement_mapping.get(item.lower(), item.lower().title())
        for item in sup_list if isinstance(item, str)
    )) if isinstance(sup_list, list) else []
)
df["supplements"].head()

In [ ]:
# 1. 'Explode' the lists so each supplement gets its own row
exploded_supps = df["supplements"].explode()

print(f"Total unique supplements: {exploded_supps.nunique()}")
print("\nTop mentioned supplements:")

pd.set_option('display.max_rows', None)
display(exploded_supps.str.lower().value_counts())

In [ ]:
df.columns

In [ ]:
# Set the option to display full column content
pd.set_option('display.max_colwidth', None)

# 1. High-Level Statistics

A high-level overview of the extracted data.
* **Classification:** The final label (positive, negative, ineffective, etc.)
* **Intent:** The sentence's purpose (experience, question, etc.)
* **Subreddit:** The source of the comment/submission.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 24))
    
# 1. Classification Distribution
class_data = df['classification']
sns.countplot(
    y=class_data,
    ax=axes[0],
    orient='h',
    order=class_data.value_counts().index,
    palette="viridis",
    hue=class_data, 
    legend=False  
)
axes[0].set_title('Overall Sentiment Classification Distribution', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Sentence Count')
axes[0].set_ylabel('Classification')

# 2. Intent Distribution
intent_data = df['intent']
sns.countplot(
    y=intent_data,
    ax=axes[1],
    orient='h',
    order=intent_data.value_counts().index,
    palette="plasma",
    hue=intent_data, 
    legend=False     
)
axes[1].set_title('Sentence Intent Distribution', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Sentence Count')
axes[1].set_ylabel('Intent')

# 3. Subreddit Distribution
top_subreddits = df['subreddit'].value_counts().index
subreddit_data = df[df['subreddit'].isin(top_subreddits)]['subreddit']
sns.countplot(
    y=subreddit_data,
    ax=axes[2],
    orient='h',
    order=top_subreddits,
    palette="cividis",
    hue=subreddit_data, 
    legend=False      
)
axes[2].set_title('Top Subreddit Sources', fontsize=16, fontweight='bold')
axes[2].set_xlabel('Sentence Count')
axes[2].set_ylabel('Subreddit')

plt.tight_layout()
plt.show()

In [ ]:
df = df[
    ~df["classification"].isin(["neutral (question)", "neutral (non-experiential)"]) &
    (df["intent"].isin(["experience"])) &
    (np.abs(df["sentiment_score"]) > 0.2) &
    (df["intent_score"] > 0.4)
]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 12))
    
# 1. Classification Distribution
class_data = df['classification']
sns.countplot(
    y=class_data,
    ax=axes[0],
    orient='h',
    order=class_data.value_counts().index,
    palette="viridis",
    hue=class_data,  
    legend=False  
)
axes[0].set_title('Overall Sentiment Classification Distribution', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Sentence Count')
axes[0].set_ylabel('Classification')

# 3. Subreddit Distribution
top_subreddits = df['subreddit'].value_counts().index
subreddit_data = df[df['subreddit'].isin(top_subreddits)]['subreddit']
sns.countplot(
    y=subreddit_data,
    ax=axes[1],
    orient='h',
    order=top_subreddits,
    palette="cividis",
    hue=subreddit_data, 
    legend=False      
)
axes[1].set_title('Top Subreddit Sources', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Sentence Count')
axes[1].set_ylabel('Subreddit')

plt.tight_layout()
plt.show()

# 2. Data "Explosion" for Deeper Analysis
To analyze per-supplement and per-aspect, we need to "explode" the list columns.

1.  `df_supps`: One row per **(sentence, supplement)**. This is used for frequency and sentiment analysis.
2.  `df_aspects`: One row per **(sentence, aspect)**.
3.  `df_golden`: One row per **(sentence, supplement, aspect)**. This is the most granular and powerful dataframe.

In [ ]:
# Creates one row for each supplement mentioned in a sentence
df_supps = df.explode('supplements').reset_index(drop=True)
df_supps = df_supps.dropna(subset=['supplements'])
df_supps['supplements'] = df_supps['supplements'][df_supps['supplements']!= "Supplement"]
print(f"Created df_supps: {len(df_supps):,} (sentence, supplement) pairs.")

# This is a two-step process:
# 1. Explode the list of aspect dictionaries
# 2. Normalize the dictionary into separate 'aspect' and 'polarity' columns
df_aspects = df.explode('aspects').dropna(subset=['aspects'])

# Check if 'aspects' column contains data before normalization

aspect_data = pd.json_normalize(df_aspects['aspects'])

# Reset index to allow for a clean join
df_aspects = df_aspects.drop('aspects', axis=1).reset_index(drop=True)
aspect_data = aspect_data.reset_index(drop=True)

df_aspects = df_aspects.join(aspect_data)
print(f"Created df_aspects: {len(df_aspects):,} (sentence, aspect) pairs.")

# Explode supplements from the already-exploded-by-aspect DataFrame
# This gives us (sentence, supplement, aspect)
df_golden = df_aspects.explode('supplements').reset_index(drop=True)
df_golden = df_golden.dropna(subset=['supplements', 'aspect'])
print(f"Created df_golden: {len(df_golden):,} (sentence, supplement, aspect) triples.")
    
print("\n'df_supps' (per-supplement) head:")
display(df_supps.head())
print("\n'df_golden' (per-supplement, per-aspect) head:")
display(df_golden.head())

# 3. Supplement Frequency Analysis

Now we use `df_supps` to see which supplements are discussed most frequently.

In [ ]:
plt.figure(figsize=(12, 10))
    
supp_counts = df_supps['supplements'].value_counts().nlargest(25)

sns.barplot(
    x=supp_counts.values,
    y=supp_counts.index,
    palette="rocket",
    orient='h',
    hue=supp_counts.index,
    legend=False    
)

plt.title('Top 25 Most Mentioned Supplements', fontsize=18, fontweight='bold')
plt.xlabel('Number of Sentences')
plt.ylabel('Supplement')
plt.show()

# 4. Supplement Sentiment Analysis

This is the core of the analysis. We'll find the sentiment profile for each supplement.

* We calculate the *percentage* of positive, negative, and ineffective mentions, as raw counts can be misleading.
* We set a **minimum mention threshold** (e.g., 100 sentences) to ensure statistical significance and avoid noise from rarely-mentioned items.

In [ ]:
df_exp = df_supps.copy()

# Set a minimum mention threshold
MIN_MENTIONS = 100

In [ ]:
# Group by supplement and classification
supp_sentiment = df_exp.groupby('supplements')['classification'].value_counts().unstack(fill_value=0)

# Calculate total experiential mentions
supp_sentiment['total'] = supp_sentiment.sum(axis=1)

# Filter for supplements meeting the minimum threshold
supp_sentiment_min = supp_sentiment[supp_sentiment['total'] >= MIN_MENTIONS].copy()

print(f"Analyzing {len(supp_sentiment_min)} supplements with >= {MIN_MENTIONS} experiential mentions.")

# Normalize to percentages
supp_sentiment_norm = supp_sentiment_min.drop('total', axis=1).div(supp_sentiment_min['total'], axis=0) * 100

# Combine 'negative' and 'ineffective' for a total "negative outcome" score
supp_sentiment_norm['negative_total'] = supp_sentiment_norm['negative'] + supp_sentiment_norm['ineffective']

In [ ]:
plt.figure(figsize=(12, 7))
top_10_pos = supp_sentiment_norm.sort_values('positive', ascending=False).head(15)
sns.barplot(
    x=top_10_pos['positive'],
    y=top_10_pos.index,
    palette="summer",
    orient='h',
    hue=top_10_pos.index, # <-- Set hue to the y-variable
    legend=False         # <-- Hide the legend
)
plt.title(f'Top 15 Most Positively-Discussed Supplements (min {MIN_MENTIONS} mentions)', fontsize=16, fontweight='bold')
plt.xlabel('Percent of Mentions classified as "Positive"')
plt.ylabel('Supplement')
plt.xlim(0, 100)
plt.show()

In [ ]:
print(top_10_pos)

In [ ]:
plt.figure(figsize=(12, 7))
top_10_neg = supp_sentiment_norm.sort_values('negative', ascending=False).head(10)
sns.barplot(
    x=top_10_neg['negative'],
    y=top_10_neg.index,
    palette="autumn",
    orient='h',
    hue=top_10_neg.index,
    legend=False 
)
plt.title(f'Top 10 Most Negatively-Discussed (min {MIN_MENTIONS} mentions)', fontsize=16, fontweight='bold')
plt.xlabel('Percent of Mentions classified as "Negative"')
plt.ylabel('Supplement')
plt.xlim(0, 100)
plt.show()

In [ ]:
top_20_frequent_names = supp_counts.head(20).index

plot_data = supp_sentiment_norm.loc[supp_sentiment_norm.index.isin(top_20_frequent_names)]

# Re-order the index based on the original frequency for a logical plot
plot_data = plot_data.reindex(top_20_frequent_names.intersection(plot_data.index))
plot_data = plot_data.iloc[::-1]

plot_data[['positive', 'negative', 'ineffective']].plot(
    kind='barh',
    stacked=True,
    figsize=(14, 12),
    color=sns.color_palette(["#2ecc71", "#e74c3c", "#f1c40f", "#f1c40f"]) # Green, Red, Yellow
)
plt.title(f'Sentiment Profile of Top-Mentioned Supplements (min {MIN_MENTIONS} mentions)', fontsize=18, fontweight='bold')
plt.xlabel('Percentage of Experiential Mentions')
plt.ylabel('Supplement')
plt.legend(title='Classification')
plt.tight_layout()
plt.show()

# 5. Stack Analysis (Co-mentions)
What supplements are frequently mentioned *together* in the same sentence? This helps us find popular "stacks". We'll look for pairs (2-combinations).

In [ ]:
# Filter for rows where 'supplements' is a list and has 2+ items
df_stacks = df[df['supplements'].apply(lambda x: isinstance(x, list) and len(x) > 1)].copy()

supplement_mapping = {
    "Magnesium": ["Magnesium Glycinate", "Magnesium Citrate", "Magnesium Threonate"]
}
reverse_supplement_mapping = {
    specific: general
    for general, specifics_list in supplement_mapping.items()
    for specific in specifics_list
}

# Items to exclude from our final lists
exclude_items = {"Nootropic (General)", "Supplement"}

def clean_and_normalize_list(sup_list):
    """
    Normalizes supplements and filters out excluded items.
    Returns a list of unique, cleaned supplement names.
    """
    if not isinstance(sup_list, list):
        return []
    
    cleaned_set = set()
    for item in sup_list:
        if isinstance(item, str):
            # 1. Normalize the item
            normalized_item = reverse_supplement_mapping.get(item, item.title())
            
            # 2. Add to set *only if* it's not in the exclude list
            if normalized_item not in exclude_items:
                cleaned_set.add(normalized_item)
                
    return list(cleaned_set)

# Apply this function to create a new, correct column
df_stacks['cleaned_supplements'] = df_stacks['supplements'].apply(clean_and_normalize_list)

# After cleaning, some lists might have < 2 items. Filter them out.
df_stacks = df_stacks[df_stacks['cleaned_supplements'].apply(len) > 1].copy()

print(f"Found {len(df_stacks):,} sentences with 2+ cleaned supplements.")

stack_counts = Counter()

# Iterate over the new 'cleaned_supplements' column
for supp_list in df_stacks['cleaned_supplements']:
    # Sort the list to treat (A, B) and (B, A) as the same pair
    # The list items are already unique from the set() in the function
    sorted_list = sorted(supp_list) 
    
    # Find all 2-supplement combinations
    for combo in combinations(sorted_list, 2):
        stack_counts[combo] += 1
        
print("\n--- Top 20 Most Common Supplement Pairs ---")

# Convert to a DataFrame for easy display
stack_df = pd.DataFrame(stack_counts.most_common(20), columns=['Pair', 'Count'])

# Format the 'Pair' column for better readability
stack_df['Pair'] = stack_df['Pair'].apply(lambda x: f"{x[0]} + {x[1]}")

display(stack_df)

In [ ]:
plt.figure(figsize=(12, 10))
sns.barplot(
    x=stack_df['Count'],
    y=stack_df['Pair'],
    palette="coolwarm",
    orient='h',
    hue=stack_df['Pair'], 
    legend=False    
)
plt.title('Top 20 Most Common Supplement Pairs (Co-mentions)', fontsize=16, fontweight='bold')
plt.xlabel('Number of Co-mentions')
plt.ylabel('Supplement Pair')
plt.show()

# 6. Aspect-Based Analysis
Now we use the `df_golden` DataFrame to find connections between supplements and specific aspects (e.g., "cognition", "sleep", "side_effects").

This lets us answer questions like:
1.  What are the most-discussed aspects overall?
2.  For "Ashwagandha", what aspects are discussed most?
3.  For "side_effects", which supplements are mentioned most?

In [ ]:
plt.figure(figsize=(12, 7))
aspect_counts = df_golden['aspect'].value_counts()
sns.barplot(
    x=aspect_counts.values,
    y=aspect_counts.index,
    palette="crest",
    orient='h',
    hue=aspect_counts.index,
    legend=False     
)
plt.title('Most-Discussed Aspects (Overall)', fontsize=16, fontweight='bold')
plt.xlabel('Number of Mentions')
plt.ylabel('Aspect')
plt.show()

In [ ]:

MIN_POS_COUNT = 60
MIN_NEG_COUNT = 0
FONT_SIZE_ANNOT = 9  


# --- 1. Filter data (Same logic as before) ---
df_filtered = df_golden[~df_golden['aspect'].isin(['side_effects', "taste"])].copy()
df_filtered = df_filtered[~df_filtered['supplements'].isin(["Supplement"])]
df_filtered = df_filtered[df_filtered['polarity'].isin(['positive', 'negative'])]

# --- 2. Calculate counts ---
polarity_counts = df_filtered.groupby(['supplements', 'aspect', 'polarity']).size().unstack(fill_value=0)

if 'positive' not in polarity_counts.columns: polarity_counts['positive'] = 0
if 'negative' not in polarity_counts.columns: polarity_counts['negative'] = 0

# --- 3. Find valid cells (Strict Threshold) ---
valid_cells_mask = (
    (polarity_counts['positive'] > MIN_POS_COUNT) & 
    (polarity_counts['negative'] > MIN_NEG_COUNT)
)
valid_cell_index = polarity_counts[valid_cells_mask].index


print(f"Found {len(valid_cell_index)} highly-debated pairs.")

# --- 4. Calculate mean polarity ---
df_valid = df_filtered[df_filtered.set_index(['supplements', 'aspect']).index.isin(valid_cell_index)].copy()
df_valid['polarity_score'] = df_valid['polarity'].map({'positive': 1, 'negative': -1})

heatmap_data = df_valid.groupby(['supplements', 'aspect'])['polarity_score'].mean()

# Pivot: Aspects on Index (Vertical), Supplements on Columns
heatmap_pivot = heatmap_data.unstack(level='supplements')

# Drop empty rows/cols
heatmap_pivot = heatmap_pivot.dropna(how='all', axis=0).dropna(how='all', axis=1)

# --- 5. Sorting (Clustering) ---
heatmap_pivot = heatmap_pivot.loc[
    heatmap_pivot.mean(axis=1).sort_values(ascending=False).index,
    heatmap_pivot.mean(axis=0).sort_values(ascending=False).index
]

# --- 6. Interactive Plotting with Plotly ---

# Determine chart size dynamically
n_rows, n_cols = heatmap_pivot.shape
height_px = max(500, n_rows * 30)
width_px = max(600, n_cols * 50)

fig = px.imshow(
    heatmap_pivot,
    labels=dict(x="Supplement", y="Aspect", color="Sentiment"),
    x=heatmap_pivot.columns,
    y=heatmap_pivot.index,
    text_auto='.2f',                # Show the numbers, formatted to 2 decimals
    aspect="auto",                  # Allow rectangular cells (not forced squares)
    color_continuous_scale='RdBu',  # Red (Neg) to Blue (Pos)
    range_color=[-1, 1]             # Lock scale from -1 to 1
)

# Customizing the visual style
fig.update_traces(
    textfont_size=FONT_SIZE_ANNOT,  # Goal 1: Smaller font
    # Goal 2: Custom Hover Tooltip
    hovertemplate="<b>Aspect:</b> %{y}<br><b>Supplement:</b> %{x}<br><b>Score:</b> %{z:.2f}<extra></extra>"
)

fig.update_layout(
    title={
        'text': f"<b>High-Controversy Map</b><br><sup>(Pairs with >{MIN_POS_COUNT} Pos & >{MIN_NEG_COUNT} Neg Reviews)</sup>",
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    width=width_px,
    height=height_px,
    xaxis={'side': 'bottom', 'tickangle': 45}, # Rotate x-labels
    yaxis={'side': 'left'},
    coloraxis_colorbar=dict(
        title="Score",
        tickvals=[-1, 0, 1],
        ticktext=["Neg (-1)", "Neutral", "Pos (+1)"]
    )
)

fig.show()

Aspect Polarity Breakdown for a Specific Supplement

In [ ]:
# ==================================
# Interactive Supplement Aspect Breakdown
# ==================================

# 1. Setup Supplement Options
# Get unique supplements, sort them, and remove generic "Supplement" entries
supp_options = sorted([x for x in df_golden['supplements'].unique() if x != "Supplement"])

# 2. Define the Plotting Function
def plot_supplement_breakdown(selected_supp, min_aspect_mentions):
    
    # --- Filter Data for Selected Supplement ---
    supp_aspects = df_golden[df_golden['supplements'] == selected_supp].copy()
    
    if supp_aspects.empty:
        print(f"No data found for {selected_supp}")
        return

    # --- Aggregate Data ---
    # Group by aspect and count polarity
    aspect_polarity_counts = supp_aspects.groupby('aspect')['polarity'].value_counts().unstack(fill_value=0)

    # --- Prepare Colors & Order ---
    # Using Seaborn's 'deep' palette: Green (index 2) for Positive, Red (index 3) for Negative
    palette = sns.color_palette("deep", 10)
    polarity_order = ['positive', 'negative']
    color_map = {
        'positive': palette[2], 
        'negative': palette[3]
    }

    # Reindex to ensure columns exist even if data is missing one polarity
    plot_data = aspect_polarity_counts.reindex(columns=polarity_order, fill_value=0)

    # --- Filter by Minimum Mentions (New Feature) ---
    # Calculate total before sorting
    plot_data['total'] = plot_data.sum(axis=1)
    
    # Filter out aspects that don't meet the slider threshold
    plot_data = plot_data[plot_data['total'] >= min_aspect_mentions]

    if plot_data.empty:
        print(f"No aspects found for {selected_supp} with >= {min_aspect_mentions} mentions.")
        return

    # Sort by total mentions (descending)
    plot_data = plot_data.sort_values('total', ascending=False)

    # --- Assign Colors dynamically based on remaining columns ---
    # (This prevents errors if a supplement has ONLY positive or ONLY negative reviews)
    current_cols = [c for c in polarity_order if c in plot_data.columns]
    plot_colors = [color_map[col] for col in current_cols]

    # --- Plotting ---
    fig, ax = plt.subplots(figsize=(14, max(6, len(plot_data) * 0.4))) # Dynamic height based on # of aspects

    # Plot (dropping total col for the bar chart)
    plot_data.drop('total', axis=1).plot(
        kind='barh',
        stacked=True,
        color=plot_colors,
        width=0.75,
        ax=ax
    )

    # --- Add Total Count Annotations ---
    max_val = plot_data['total'].max()
    
    for i, total in enumerate(plot_data['total']):
        ax.text(
            total + (max_val * 0.01), 
            i, 
            f"{int(total)}", 
            ha='left', va='center',
            fontsize=11, color='black', fontweight='bold'
        )

    # --- Styling ---
    ax.set_title(f'Aspect Polarity Breakdown: {selected_supp}\n(Min Mentions per Aspect: {min_aspect_mentions})', 
                 fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Number of Mentions', fontsize=13)
    ax.set_ylabel('Aspect', fontsize=13)

    ax.invert_yaxis() # Most discussed at top
    ax.set_xlim(0, max_val * 1.15) # Add breathing room for labels

    # Legend formatting
    ax.legend(title='Polarity', bbox_to_anchor=(1.01, 1), loc='upper left')
    
    sns.despine(top=True, right=True)
    plt.tight_layout()
    plt.show()

# 3. Create Widgets
style = {'description_width': 'initial'}

# Dropdown for Supplement Selection
supp_dropdown = widgets.Dropdown(
    options=supp_options,
    value=supp_options[0] if supp_options else None,
    description='Select Supplement:',
    style=style,
    layout={'width': '400px'}
)

# Slider to filter out "noise" (rarely mentioned aspects)
min_mentions_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=20,
    step=1,
    description='Min Mentions per Aspect:',
    style=style,
    continuous_update=False
)

# 4. Run Interaction
interact(plot_supplement_breakdown, 
         selected_supp=supp_dropdown, 
         min_aspect_mentions=min_mentions_slider);

In [ ]:
# ==================================
# Interactive Supplement Sentiment Analyzer
# ==================================

# 1. Setup Options for the Dropdown
# Get unique aspects from your data and sort them
available_aspects = sorted(df_golden['aspect'].unique().tolist())
# Add an 'OVERALL' option to the beginning of the list
options_list = ['OVERALL'] + available_aspects

# 2. Define the Plotting Function
def plot_sentiment(selected_aspects, min_mentions):
    # --- A. Filter Data based on Aspect Selection ---
    # Check if user selected 'OVERALL' or nothing, otherwise filter by specific aspects
    if 'OVERALL' in selected_aspects or not selected_aspects:
        # Use the whole dataframe, but remove generic "Supplement" rows
        filtered_df = df_golden.copy()
        title_aspect = "OVERALL (All Aspects)"
    else:
        # Filter for only the selected aspects
        filtered_df = df_golden[df_golden['aspect'].isin(selected_aspects)].copy()
        # Create a dynamic title string
        if len(selected_aspects) > 3:
            title_aspect = f"{len(selected_aspects)} Selected Aspects"
        else:
            title_aspect = ", ".join(selected_aspects)

    # Remove generic names if present
    filtered_df = filtered_df[~filtered_df['supplements'].isin(["Supplement"])]

    # --- B. Convert Polarity to Numeric Score ---
    filtered_df['sentiment_score'] = filtered_df['polarity'].map({'positive': 1, 'negative': -1})

    # --- C. Aggregate by Supplement ---
    supp_stats = filtered_df.groupby('supplements')['sentiment_score'].agg(['mean', 'count'])

    # --- D. Filter and Sort by Min Mentions ---
    qualified_supps = supp_stats[supp_stats['count'] >= min_mentions]
    
    # Handle empty data case (if filter is too strict)
    if qualified_supps.empty:
        print(f"No supplements found with >= {min_mentions} mentions for this selection.")
        return

    # Sort by highest average sentiment
    plot_data = qualified_supps.sort_values(by='mean', ascending=False).head(20)

    # --- E. Plotting (Your Original Visual Logic) ---
    plt.figure(figsize=(12, 10))

    # Color palette logic
    norm = plt.Normalize(-1, 1)
    colors = plt.cm.RdBu(norm(plot_data['mean'].values))

    ax = sns.barplot(
        x=plot_data['mean'],
        y=plot_data.index,
        palette=colors,
        orient='h'
    )

    # Dynamic Title
    plt.title(f'Best Rated Supplements for: {title_aspect}\n(Min Mentions: {min_mentions})', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Average Sentiment Score (-1 = Negative, +1 = Positive)', fontsize=13)
    plt.ylabel('Supplement', fontsize=13)

    # Vertical line at 0
    plt.axvline(0, color='black', linewidth=1, linestyle='--')

    # Text labels
    for i, (score, count) in enumerate(zip(plot_data['mean'], plot_data['count'])):
        text_x = score + 0.02 if score >= 0 else score - 0.02
        ha = 'left' if score >= 0 else 'right'
        
        ax.text(
            text_x, i, 
            f"{score:.2f} (n={count})", 
            va='center', ha=ha,
            fontsize=11, fontweight='bold', color='#333333'
        )

    # Adjust limits
    if not plot_data.empty:
        lower_lim = min(-1.1, plot_data['mean'].min() - 0.2)
        upper_lim = max(1.1, plot_data['mean'].max() + 0.2)
        plt.xlim(lower_lim, upper_lim)

    sns.despine(left=True, bottom=False)
    plt.tight_layout()
    plt.show()

# 3. Create the Widgets and Display
style = {'description_width': 'initial'}

aspect_selector = widgets.SelectMultiple(
    options=options_list,
    value=['OVERALL'], # Default value
    description='Select Aspect(s):',
    style=style,
    rows=10,
    disabled=False
)

mention_slider = widgets.IntSlider(
    value=20,
    min=1,
    max=100,
    step=1,
    description='Min Mentions:',
    style=style,
    continuous_update=False # Only plot when user lets go of slider handle
)

# 4. Run the Interaction
interact(plot_sentiment, selected_aspects=aspect_selector, min_mentions=mention_slider);

In [ ]:
# 1. Define the Explorer Function
def explore_sentences(supplement, aspect, polarity, n_records):
    
    # --- A. Filter the Data ---
    filtered_df = df_golden[
        (df_golden['supplements'] == supplement) & 
        (df_golden['aspect'] == aspect) & 
        (df_golden['polarity'] == polarity)
    ]
    
    # --- B. Handle Empty Results ---
    if filtered_df.empty:
        print(f"No records found for:\nSupp: {supplement}\nAspect: {aspect}\nPolarity: {polarity}")
        return

    # --- C. Display Output ---
    count = len(filtered_df)
    
    # Use HTML for the header to make it stand out
    display(HTML(f"<h3>Found {count} records. Showing top {min(n_records, count)}:</h3>"))
    
    unique_sentences = filtered_df['sentence_context'].unique()
    
    for i, sentence in enumerate(unique_sentences[:n_records]):
        text = sentence.strip()
        
        # --- HIGHLIGHTING LOGIC ---
        # We use regex to find the supplement name (case-insensitive) 
        # and wrap it in a yellow highlight span.
        # re.escape ensures special characters in supplement names (like + or ()) don't break regex.
        pattern = re.escape(supplement)
        
        # The lambda function ensures we keep the original casing of the sentence text 
        # but apply the styling around it.
        highlighted_text = re.sub(
            f"({pattern})", 
            r'<span style="background-color: #ffd700; font-weight: bold; padding: 0 4px;">\1</span>', 
            text, 
            flags=re.IGNORECASE
        )
        
        # --- RENDERING ---
        # We create an HTML string. 
        # 'white-space: pre-wrap' ensures lines wrap within the cell width 
        # but preserves original paragraphs.
        html_output = f"""
        <div style="border: 1px solid #ddd; padding: 15px; margin-bottom: 10px; border-radius: 5px;">
            <div style="font-weight: bold; color: #555; margin-bottom: 5px;">Record #{i+1}</div>
            <div style="font-size: 14px; line-height: 1.6; white-space: pre-wrap;">{highlighted_text}</div>
        </div>
        """
        
        display(HTML(html_output))

# 2. Setup Widgets

supp_widget = widgets.Dropdown(
    options=sorted(df_golden['supplements'].unique()),
    description='Supplement:',
    style={'description_width': 'initial'}
)

aspect_widget = widgets.Dropdown(
    options=sorted(df_golden['aspect'].unique()),
    description='Aspect:',
    style={'description_width': 'initial'}
)

polarity_widget = widgets.Dropdown(
    options=['positive', 'negative'],
    value='positive',
    description='Polarity:',
    style={'description_width': 'initial'}
)

n_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=50,
    step=1,
    description='Num Records:',
    style={'description_width': 'initial'}
)

# 3. Run Interaction
interact(explore_sentences, 
         supplement=supp_widget, 
         aspect=aspect_widget, 
         polarity=polarity_widget, 
         n_records=n_slider);